# Script to Convert File from GeoTiff to NetCDF
**Input Data:** GeoTiff file  
**Output Data:** NetCDF file  
**Description:** Extracts data from a GeoTiff (.tif) file and exports them to a new NetCDF(.nc) file.  
**Date:** July 2022  
**Creator:** Emma Perkins 

In [ ]:
# PARAMETERS

# tif input path and file
tif_in_path = '/glade/scratch/eperkins/data/'
tif_in_file = 'cmc_sdepth_dly_2000_v01.2.tif'

# netcdf output path and file
nc_out_path = '/glade/scratch/eperkins/data/tif_to_nc/'
nc_out_file = 'corr_cmc_sdepth_dly_1998_v01.2.nc'

In [13]:
re = 6378.137  # Earth Radius
e = 0.01671  # Earth eccentricity
hemisphere = SOUTH
true_scale_lat = 70  # true-scale latitude in degrees

### Install Important Packages
More information about the NSIDC package for converting from polar stereographic projection to standard lat/lon coordinates can be found at: https://github.com/nsidc/polarstereo-lonlat-convert-py

In [1]:
# install NSIDC package for converting from polar stereographic projection to lat/lon coordinates
!pip install /glade/scratch/eperkins/coordinate_convert/polarstereo_lonlat_convert_py/

Processing /glade/scratch/eperkins/coordinate_convert/polarstereo_lonlat_convert_py
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for polarstereo-lonlat-convert-py: filename=polarstereo_lonlat_convert_py-1.0.0-py3-none-any.whl size=6274 sha256=27e00a8dac7ac4a617428030a641a53dff8244fe294e93cf8d821a1a0086b839
  Stored in directory: /glade/scratch/eperkins/pip-ephem-wheel-cache-lp5051p1/wheels/27/20/52/2db6fab80c2735aaf2030ad714b62073abafefb318d4d00673
Successfully built polarstereo-lonlat-convert-py
  Attempting uninstall: polarstereo-lonlat-convert-py
    Found existing installation: polarstereo-lonlat-convert-py 1.0.0
    Uninstalling polarstereo-lonlat-convert-py-1.0.0:
      Successfully uninstalled polarstereo-lonlat-convert-py-1.0.0


In [ ]:
# import relevant packages
import xarray as xr
from osgeo import gdal
import numpy as np
import datetime
from polar_convert.constants import NORTH
from polar_convert import polar_xy_to_lonlat
from polar_convert.constants import SOUTH

### Load Files

In [ ]:
data = xr.open_rasterio(tif_in_path+tif_in_file)

/glade/work/eperkins/miniconda3/envs/analysis3/lib/python3.7/site-packages/ipykernel_launcher.py:4: DeprecationWarning: open_rasterio is Deprecated in favor of rioxarray. For information about transitioning, see: https://corteva.github.io/rioxarray/stable/getting_started/getting_started.html
  after removing the cwd from sys.path.


### Convert Spatial Coordinates to Standard Lat/Lon

In [14]:
x_coords = data.x.to_numpy()
y_coords = data.y.to_numpy()

In [15]:
new_lon, new_lat = polar_xy_to_lonlat(x_coords, y_coords, true_scale_lat, re, e, hemisphere)

In [4]:
data = data.rename({'band': 'time', 'x': 'lon', 'y': 'lat'})
new_data = np.zeros([len(data.time), len(data.lat), len(data.lon)])

In [ ]:
new_lat = np.zeros(len(data.y))
new_lon = np.zeros(len(data.x))
new_lon, new_lat = polar_xy_to_lonlat(data.x, data.y, true_scale_lat, re, e, hemisphere)

time_coord = np.zeros(len(data.band))

# rename band to time
data = data.rename({'band': 'time', 'x': 'lon', 'y': 'lat'})

data['lat'] = new_lat
data['lon'] = new_lon

### Convert bands to time coordinate (each band is one day)

In [4]:
# convert bands to julian date
julian_date = np.zeros(len(data.time))
year = 99000  # replace first two digits with last two digits of specific year ex: 1999 --> 99000, 2001 --> 01000
for i in range(0, len(data.time)):
    day = int(data.time[i])
    new_day = year + day
    julian_date[i] = int(new_day)

In [ ]:
# convert julian date to datetime index
time_coord = np.empty(len(data.time), dtype=object)
for i in range(0, len(data.time)):
    day_string = str(int(julian_date[i]))
    time_coord[i] = datetime.datetime.strptime(day_string, '%y%j')

# replace time coordinate with new datetime array
data['time'] = time_coord

### Convert to NetCDF File and Export

In [ ]:
gdal.Translate(nc_out_path+nc_out_file, nc_out_path+nc_out_file, format='NetCDF')

<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x2ad39ea17840> >